## CUSTOM GURADRAILS - BEFORE AGENT 

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware,AgentState,hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """ Deterministic guardrails : Block requests containing banned keywords
    this runs before the agent is called
    """
    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]
        
    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState , runtime: Runtime) -> dict[str,Any]:
        if not state["messages"]:
            return None
        first_message = state["messages"][0]
        if first_message.type != "human":
            return None
        content = first_message.content.lower()
        
        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked request with keyword {keyword}")
                return{
                    "messages":[{
                        "role":"assistant",
                        "content":(
                            "I cannot process requests containing banned keywords."
                            "please rephrase your request."
                        )
                    }],
                    "jump_to":"end"
                }
        return None
    
@tool
def search_tool(query :str):
    """Search for information based on the user's query."""
    return f"results for {query}"
    
filterd_agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    tools=[search_tool],
    middleware=[(
        
        ContentFilterMiddleware(banned_keywords=["hack","exploit","malware","jailbreak","bypass"])
        )
        ]
    )
print("ContentFilterMiddleware agent is ready")

ContentFilterMiddleware agent is ready


In [19]:
res1 = filterd_agent.invoke({
    "messages":[{"role":"user",
                 "content":"what is machine learning"
                 }]
})
res1

{'messages': [HumanMessage(content='what is machine learning', additional_kwargs={}, response_metadata={}, id='3053c0a2-4c1c-45d5-a507-d730bbfa79a4'),
  AIMessage(content='**Machine Learning (ML)** is a subfield of artificial intelligence (AI) that focuses on building systems that can **learn from data** rather than being explicitly programmed for every task. Instead of using a set of rigid, hand-coded rules, an ML algorithm is fed data and uses statistical methods to identify patterns, make predictions, or make decisions.\n\n### Core Concepts\n1. **Data**: The "fuel" for ML. It consists of examples (e.g., images, text, numbers) that the model uses to learn.\n2. **Model**: A mathematical function that maps input data to output predictions.\n3. **Training**: The process where the model is adjusted (its parameters are updated) to minimize errors between its predictions and the actual outcomes in the training data.\n4. **Generalization**: The ability of the model to perform well on new, u

In [ ]:
res2 = filterd_agent.invoke({
    "messages":[{
        "role" : "user",
        "content" : "how to hack my phone"
    }]
})
res2

Blocked request with keyword hack


{'messages': [HumanMessage(content='how to hack my phone', additional_kwargs={}, response_metadata={}, id='53dec1bf-1794-48be-b84d-df4b17a98151'),
  AIMessage(content='I cannot process requests containing banned keywords.please rephrase your request.', additional_kwargs={}, response_metadata={}, id='b38c7365-376e-46af-8f1e-85ba132875f0', tool_calls=[], invalid_tool_calls=[])]}